# TensorCircuit SDK 对接天衍量子云

本教程沿用 TensorCircuit 云 SDK 的 provider-agnostic 工作流，通过统一的 `tc.cloud.apis` 入口连接中电信天衍量子计算云平台。内容覆盖设备查询、任务生命周期、云模拟器、批量提交，以及真实量子机的拓扑校验。

天衍平台提供 `tianyan_sw` 等模拟器和 `tianyan176` 等真实量子设备，原生任务格式为 QCIS。设备名称和在线状态以平台实时返回为准。

## 环境准备

```bash
pip install "tensorcircuit-ng[cloud]"
pip install "cqlib>=1.3.10,<1.4"
```

`cqlib` 需要 Python 3.10 或更高版本以及 NumPy 2.1.2 或更高版本。登录密钥应通过 `TC_TOKEN_TIANYAN` 环境变量提供（云 SDK 的 token 系统会自动读取），不要写入 Notebook。真实设备代码还需要显式设置 `TIANYAN_RUN_HARDWARE=1`。

In [ ]:
import os

if not os.getenv("TC_TOKEN_TIANYAN"):
    raise RuntimeError("请先设置 TC_TOKEN_TIANYAN 环境变量")

## Provider 配置

与 Tencent 教程一致，云平台操作统一从 `tc.cloud.apis` 进入。切换 provider 后，设备、任务和结果对象仍使用同一套抽象。Token 只保存在当前进程中，不写入本地缓存。

In [ ]:
tc.cloud.apis.set_provider("tianyan")
# token is picked up automatically from the TC_TOKEN_TIANYAN env var

print("Providers:", tc.cloud.apis.list_providers())
print("Current provider:", tc.cloud.apis.get_provider())

## Devices and properties

先查询平台实时设备列表，再选择模拟器或真实量子机。`Device` 对象提供属性、原生门集合和耦合拓扑等统一接口。

In [ ]:
devices = tc.cloud.apis.list_devices(provider="tianyan")
print(f"Available TianYan devices ({len(devices)}):")
for available_device in devices:
    print(f"  - {available_device}")

sim_device = tc.cloud.apis.get_device("tianyan::tianyan_sw")
real_device = tc.cloud.apis.get_device("tianyan::tianyan176")

真实设备的校准数据和耦合关系决定电路能否直接映射。下面只查询设备元数据，不提交真实硬件任务。

In [ ]:
properties = real_device.list_properties()
print("Native gates:", real_device.native_gates())
print("Available qubits:", len(properties["bits"]))
print("Coupling edges:", len(properties["links"]) // 2)
print("First topology edges:", real_device.topology()[:10])

## Tasks

下面构造 Bell 态并提交到云模拟器。TensorCircuit 电路会转换为 QCIS；测量指令按记录顺序作为末端测量提交。

In [ ]:
def bell_circuit():
    circuit = tc.Circuit(2)
    circuit.h(0)
    circuit.cx(0, 1)
    circuit.measure_instruction(0, 1)
    return circuit


circuit = bell_circuit()
task = tc.cloud.apis.submit_task(circuit=circuit, device=sim_device, shots=1000)
print("Task ID:", task.id_)

`results(blocked=True)` 会等待任务结束并返回 counts。理想 Bell 态主要出现 `00` 和 `11`。

In [ ]:
counts = task.results(blocked=True)
print("Counts:", counts)
tc.results.counts.plot_histogram(counts)

与 Tencent 的 `Task` 工作流相同，可以查询状态、详情和任务 ID。TianYan 详情额外保留提交的 QCIS 源码。已有任务也可以用 ID 和设备重新构造 `Task` 对象。

In [ ]:
details = task.details()
print("State:", task.state())
print("Shots:", details["shots"])
print("QCIS:\n", details["source"])

restored_task = tc.cloud.apis.get_task(task.id_, device=sim_device)
print("Restored task result:", restored_task.results(blocked=True))

## Cloud simulator

模拟器支持批量提交。与 Tencent 接口相同，电路列表会返回对应的 `Task` 列表；同一批次要求使用相同的 shots。

In [ ]:
circuits = [bell_circuit() for _ in range(3)]
tasks = tc.cloud.apis.submit_task(circuit=circuits, device=sim_device, shots=100)

for index, batch_task in enumerate(tasks, start=1):
    print(f"Task {index}: {batch_task.results(blocked=True)}")

已有 QCIS 或 OpenQASM 2 源码时，可以绕过 TensorCircuit 电路构造。OpenQASM 会先由 `cqlib` 转换为 QCIS。

In [ ]:
qcis = "H Q0\nH Q1\nCZ Q0 Q1\nH Q1\nM Q0\nM Q1"
qcis_task = tc.cloud.apis.submit_task(source=qcis, device=sim_device, shots=100)
print("Direct QCIS result:", qcis_task.results(blocked=True))

In [ ]:
qasm = """OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
cx q[0],q[1];
measure q -> c;
"""
qasm_task = tc.cloud.apis.submit_task(
    source=qasm, lang="OPENQASM", device=sim_device, shots=100
)
print("OpenQASM result:", qasm_task.results(blocked=True))

## Compiling: 门转换与拓扑校验

真实量子机受原生门集合和耦合拓扑限制。Provider 不会自动重映射比特：提交到真实量子机的电路必须已满足设备拓扑。简单场景可以用 `Circuit.initial_mapping` 把逻辑电路映射到选定的物理比特上；更复杂的电路请使用 `tensorcircuit.compiler`。TensorCircuit 电路会在提交前进行拓扑校验，不兼容时抛出 `ValueError`。真实设备会消耗平台资源，以下单元格默认只说明执行条件。

In [ ]:
if RUN_HARDWARE:
    q1, q2 = sorted(real_device.topology()[0])
    hardware_circuit = bell_circuit().initial_mapping({0: q1, 1: q2}, n=q2 + 1)
    hardware_task = tc.cloud.apis.submit_task(
        circuit=hardware_circuit, device=real_device, shots=100
    )
    hardware_counts = hardware_task.results(blocked=True)
    hardware_details = hardware_task.details()
    print("Physical qubits:", q1, q2)
    print("Hardware counts:", hardware_counts)
    print("QCIS:\n", hardware_details["source"])
else:
    print("已跳过真实设备提交；设置 TIANYAN_RUN_HARDWARE=1 后再运行")

## Provider differences and limitations

统一 API 不代表所有平台能力完全相同。当前 TianYan provider 有以下边界：

- `tc.cloud.apis.list_tasks(...)` 和 `tc.cloud.apis.remove_task(...)` 会抛出 `NotImplementedError`，因为当前 `cqlib` 没有可用的任务列表和取消端点。
- TensorCircuit 电路中的测量会作为末端测量提交，不保留中途测量语义。
- 拓扑校验仅用于真实量子机；模拟器不受硬件耦合限制。提交的电路必须事先满足设备拓扑。
- 真实设备 shots 应保持较小，并始终通过 `TIANYAN_RUN_HARDWARE=1` 显式启用。